# Data Cleaning and Preprocessing

In this notebook, the dataset is cleaned and prepared for analysis.

#### The objectives of this notebook are:

- Identify missing values
- Detect duplicate records
- Verify data types
- Convert date columns
- Perform basic data quality checks
- Create a cleaned dataset for further analysis

In [1]:
# importing libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# load datset
df = pd.read_csv("../Data/raw/global_financial_markets_2000_Now.csv")

## Missing Value Analysis

Missing values are examined to determine whether any data cleaning is required before analysis.

In [3]:
df.isnull()

,date,open,high,low,close,volume,symbol,asset_name,asset_type,region
0,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...
187606,False,False,False,False,False,False,False,False,False,False
187607,False,False,False,False,False,False,False,False,False,False
187608,False,False,False,False,False,False,False,False,False,False
187609,False,False,False,False,False,False,False,False,False,False


In [4]:
df.isnull().sum()

date          0
open          0
high          0
low           0
close         0
volume        0
symbol        0
asset_name    0
asset_type    0
region        0
dtype: int64

### Observation

The missing value analysis shows that all features contain complete information with no null entries. Since no missing values were detected across the dataset, no imputation or missing value treatment is required. This indicates a high level of data completeness and quality for further analysis.

## Duplicate Record Analysis

Duplicate records are checked to identify redundant observations that may affect the accuracy of the analysis.

In [5]:
df.duplicated()

0         False
1         False
2         False
3         False
4         False
          ...  
187606    False
187607    False
187608    False
187609    False
187610    False
Length: 187611, dtype: bool

In [6]:
df.duplicated().sum()

np.int64(0)

### Observation

No duplicate records were found in the dataset. This indicates that each observation is unique and no duplicate removal is required before further analysis.

## Date Conversion

The date column is converted to datetime format to enable efficient time-series analysis and date-based operations.

In [7]:
# before converting
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 187611 entries, 0 to 187610
Data columns (total 10 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   date        187611 non-null  str    
 1   open        187611 non-null  float64
 2   high        187611 non-null  float64
 3   low         187611 non-null  float64
 4   close       187611 non-null  float64
 5   volume      187611 non-null  int64  
 6   symbol      187611 non-null  str    
 7   asset_name  187611 non-null  str    
 8   asset_type  187611 non-null  str    
 9   region      187611 non-null  str    
dtypes: float64(4), int64(1), str(5)
memory usage: 14.3 MB


In [8]:
df["date"] = pd.to_datetime(df["date"])

In [9]:
# after converting 
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 187611 entries, 0 to 187610
Data columns (total 10 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   date        187611 non-null  datetime64[us]
 1   open        187611 non-null  float64       
 2   high        187611 non-null  float64       
 3   low         187611 non-null  float64       
 4   close       187611 non-null  float64       
 5   volume      187611 non-null  int64         
 6   symbol      187611 non-null  str           
 7   asset_name  187611 non-null  str           
 8   asset_type  187611 non-null  str           
 9   region      187611 non-null  str           
dtypes: datetime64[us](1), float64(4), int64(1), str(4)
memory usage: 14.3 MB


### Observation

The date column was successfully converted from string format to datetime format. This enables chronological sorting, date filtering, and time-series analysis in subsequent stages of the project.

## Data Consistency Checks

Data consistency checks are performed to identify any logically invalid values that may affect the reliability of the analysis.

In [10]:
(df["high"] < df["low"]).sum()

np.int64(7)

### Inconsistent Records Review

The records violating the expected condition high ≥ low were inspected manually. Only 7 observations were affected, representing less than 0.01% of the dataset.

In [11]:
df[df["high"] < df["low"]]

,date,open,high,low,close,volume,symbol,asset_name,asset_type,region
52703,2009-11-23,1164.30,1163.00,1164.30,1164.30,3,GC=F,Gold,Commodity,Global
53262,2009-12-23,75.45,75.13,75.45,75.45,1,BZ=F,Brent Crude,Commodity,Global
53274,2009-12-24,76.31,75.93,76.31,76.31,1,BZ=F,Brent Crude,Commodity,Global
53409,2010-01-04,80.12,79.82,80.12,80.12,97,BZ=F,Brent Crude,Commodity,Global
53454,2010-01-05,80.59,80.26,80.59,80.59,97,BZ=F,Brent Crude,Commodity,Global
53489,2010-01-07,81.51,81.51,81.63,81.51,7,BZ=F,Brent Crude,Commodity,Global
55257,2010-04-15,87.17,87.13,87.17,87.17,14,BZ=F,Brent Crude,Commodity,Global


### Observation

The consistency check identified 7 records where the high price is lower than the corresponding low price. These records represent a very small proportion of the dataset and are limited to commodity assets such as Gold and Brent Crude. As the number of affected records is negligible, they have been retained and documented for transparency.

In [12]:
(df["volume"] < 0).sum()

np.int64(0)

In [13]:
(df["open"] < 0).sum()

np.int64(1)

### Negative Price Investigation

Records containing negative opening prices are examined to determine whether they represent data errors or valid market events.

In [14]:
df[df["open"] < 0]

,date,open,high,low,close,volume,symbol,asset_name,asset_type,region
129637,2020-04-21,-14.0,13.86,-16.74,10.01,2288230,CL=F,Crude Oil (WTI),Commodity,Global


### Observation

One record with a negative opening price was identified during the consistency checks. Upon investigation, the record corresponds to WTI Crude Oil on 21 April 2020, a period during which oil futures temporarily traded at negative prices due to unprecedented market conditions during the COVID-19 pandemic. Therefore, the observation is considered valid market data and has been retained.

In [15]:
(df["close"] < 0).sum()

np.int64(1)

### Observation

One record with a negative closing price was identified during the consistency checks. This record corresponds to the same WTI Crude Oil event identified in the negative opening price investigation, which occurred during the historic oil market crash of April 2020. Since the negative price reflects an actual market event rather than a data quality issue, the record has been retained in the dataset.

## Saving the Cleaned Dataset

After completing the cleaning and validation steps, the processed dataset is saved for use in subsequent analysis notebooks.

In [16]:
df.to_csv(
    "../Data/processed/cleaned_market_data.csv",
    index=False
)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.


### Observation

The cleaned dataset has been successfully exported to the processed data directory. This dataset will be used as the primary data source for exploratory data analysis and subsequent market investigations.

## Cleaning Summary

The dataset was examined and prepared for further analysis through a series of data quality checks and preprocessing steps.

#### Key Actions Performed

- Missing value analysis confirmed that no null values are present in the dataset.
- Duplicate record analysis identified no duplicate observations.
- The date column was converted to datetime format for time-series analysis.
- Data types were verified to ensure consistency across all features.
- Data consistency checks revealed a small number of anomalies, which were investigated and documented.
- The cleaned dataset was exported for future analysis.

The dataset is now ready for exploratory data analysis.